### Semana 6, Dia 2

Estamos prestes a criar e usar nosso próprio MCP Server e MCP Client!

É bem simples, mas não é super simples. O entusiasmo em torno do MCP está em como é fácil compartilhar e usar outros MCP Servers — criar o nosso envolve um pouco de trabalho.

Vamos revisar um pouco de código Python feito principalmente por uma equipe de engenharia dedicada:

accounts.py


In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
from accounts import Account

In [3]:
account = Account.get("Ed")
account

Account(name='ed', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [4]:
account.buy_shares("AMZN", 3, "Porque este site de livraria parece promissor")


'Completed. Latest details:\n{"name": "ed", "balance": 9903.808, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 32.064, "timestamp": "2025-10-02 22:32:38", "rationale": "Porque este site de livraria parece promissor"}], "portfolio_value_time_series": [["2025-10-02 22:32:38", 10185.808]], "total_portfolio_value": 10185.808, "total_profit_loss": 185.8080000000009}'

In [5]:
account.report()

'{"name": "ed", "balance": 9903.808, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 32.064, "timestamp": "2025-10-02 22:32:38", "rationale": "Porque este site de livraria parece promissor"}], "portfolio_value_time_series": [["2025-10-02 22:32:38", 10185.808], ["2025-10-02 22:32:39", 9984.808]], "total_portfolio_value": 9984.808, "total_profit_loss": -15.191999999999098}'

In [6]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 32.064,
  'timestamp': '2025-10-02 22:32:38',
  'rationale': 'Porque este site de livraria parece promissor'}]

### Agora vamos escrever um MCP server e usá-lo diretamente!


In [7]:
# Agora vamos usar nosso servidor de contas como um MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [8]:
mcp_tools

[Tool(name='get_balance', description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None),
 Tool(name='get_holdings', description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None),
 Tool(name='buy_shares', description="Buy shares of a stock.\n\n    Args:\n        name: The name of the account holder\n        symbol: The symbol of the stock\n        quantity: The quantity of shares to buy\n        rationale: The rationale for the purchase and fit with the account's strategy\n    ", inputSchema={'properties': {'name': {'title': 'Name', '

In [9]:
instructions = "Você consegue administrar uma conta para um cliente e responder perguntas sobre a conta."
request = "Meu nome é Ed e minha conta está registrada como Ed. Qual é o meu saldo e quais são meus investimentos?"
model = "gpt-4.1-mini"


In [10]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Ed, seu saldo em conta é de R$ 9.903,81. Quanto aos seus investimentos, você possui 3 ações da Amazon (AMZN). Posso ajudar com mais alguma coisa?

### Agora vamos criar nosso próprio MCP Client


In [11]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None), Tool(name='get_holdings', description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None), Tool(name='buy_shares', description="Buy shares of a stock.\n\n    Args:\n        name: The name of the account holder\n        symbol: The symbol of the stock\n        quantity: The quantity of shares to buy\n        rationale: The rationale for the purchase and fit with the account's strategy\n    ", inputSchema={'properties': {'name': {'title': 'Name', 'ty

In [12]:
request = "Meu nome é Ed e minha conta está registrada como Ed. Qual é o meu saldo?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Ed, o saldo da sua conta é de R$ 9.903,81. Posso ajudar em mais alguma coisa?

In [13]:
context = await read_accounts_resource("ed")
print(context)

{"name": "ed", "balance": 9903.808, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 32.064, "timestamp": "2025-10-02 22:32:38", "rationale": "Porque este site de livraria parece promissor"}], "portfolio_value_time_series": [["2025-10-02 22:32:38", 10185.808], ["2025-10-02 22:32:39", 9984.808], ["2025-10-02 22:52:37", 10182.808]], "total_portfolio_value": 10182.808, "total_profit_loss": 182.8080000000009}


In [14]:
from accounts import Account
Account.get("ed").report()

'{"name": "ed", "balance": 9903.808, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 32.064, "timestamp": "2025-10-02 22:32:38", "rationale": "Porque este site de livraria parece promissor"}], "portfolio_value_time_series": [["2025-10-02 22:32:38", 10185.808], ["2025-10-02 22:32:39", 9984.808], ["2025-10-02 22:52:37", 10182.808], ["2025-10-02 22:52:39", 10182.808]], "total_portfolio_value": 10182.808, "total_profit_loss": 182.8080000000009}'

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercícios</h2>
            <span style="color:#ff7800;">Crie seu próprio MCP Server! Faça uma função simples para retornar a data atual e exponha-a como uma ferramenta para que um Agent possa lhe dizer a data de hoje.<br/>Exercício opcional mais difícil: depois, faça um MCP Client e use uma chamada nativa do OpenAI (sem o Agents SDK) para usar sua ferramenta por meio do seu cliente.
            </span>
        </td>
    </tr>
</table>
